In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 1000

# -----------------------------
# Categorical columns
# -----------------------------

gender = np.random.choice(
    ["Male", "Female"],
    size=n,
    p=[0.55, 0.45]
)

city = np.random.choice(
    ["Mumbai", "Pune", "Delhi", "Bangalore", "Hyderabad", "Chennai"],
    size=n
)

education = np.random.choice(
    ["High School", "Bachelor", "Master", "PhD"],
    size=n,
    p=[0.20, 0.45, 0.30, 0.05]
)

# Education -> experience tendency
education_bonus = {
    "High School": 0,
    "Bachelor": 1,
    "Master": 2,
    "PhD": 3
}

# -----------------------------
# Numerical columns
# -----------------------------

age = np.random.randint(21, 51, size=n)

# Experience related to age
experience = np.maximum(
    0,
    age - np.random.randint(20, 26, size=n)
)

# Add some education influence
experience = experience + np.array(
    [education_bonus[e] for e in education]
)

experience = np.clip(experience, 0, 30)


# Job level depends mainly on experience
job_level = np.select(
    [
        experience < 3,
        experience < 7,
        experience < 12,
        experience >= 12
    ],
    [
        "Junior",
        "Mid",
        "Senior",
        "Manager"
    ],
    default="Junior"
)


# Department
department = np.random.choice(
    ["Engineering", "Marketing", "Finance", "HR", "Sales"],
    size=n,
    p=[0.30, 0.15, 0.15, 0.10, 0.30]
)


# Performance score
performance = np.clip(
    50
    + experience * 1.2
    + np.random.normal(0, 8, n),
    40,
    100
).round(1)


# Projects completed depends on experience
projects = np.maximum(
    0,
    (experience * 1.5 + np.random.normal(0, 3, n))
).round().astype(int)


# Salary depends on:
# experience + education + job level + performance

education_salary_bonus = np.array([
    education_bonus[e] for e in education
])

job_salary_bonus = np.select(
    [
        job_level == "Junior",
        job_level == "Mid",
        job_level == "Senior",
        job_level == "Manager"
    ],
    [
        0,
        15000,
        35000,
        60000
    ]
)

salary = (
    25000
    + experience * 3500
    + education_salary_bonus * 7000
    + job_salary_bonus
    + performance * 400
    + np.random.normal(0, 5000, n)
)

salary = np.maximum(salary, 20000).round(0)


# -----------------------------
# Boolean / binary columns
# -----------------------------

has_degree = np.isin(
    education,
    ["Bachelor", "Master", "PhD"]
)

is_employed = np.ones(n, dtype=bool)


# -----------------------------
# Final DataFrame
# -----------------------------

df = pd.DataFrame({
    "Age": age,
    "Gender": gender,
    "City": city,
    "Education_Level": education,
    "Department": department,
    "Experience_Years": experience,
    "Job_Level": job_level,
    "Performance_Score": performance,
    "Projects_Completed": projects,
    "Monthly_Salary": salary,
    "Has_Degree": has_degree,
    "Is_Employed": is_employed
})

df.sample(10)


,Age,Gender,City,Education_Level,Department,Experience_Years,Job_Level,Performance_Score,Projects_Completed,Monthly_Salary,Has_Degree,Is_Employed
936,22,Male,Pune,Master,Sales,4,Mid,64.5,4,93788.0,True,True
503,25,Female,Bangalore,Bachelor,Finance,2,Junior,54.9,1,58079.0,True,True
626,47,Male,Hyderabad,Bachelor,Engineering,23,Manager,84.8,34,220726.0,True,True
899,26,Male,Pune,High School,Engineering,6,Mid,58.2,6,76383.0,False,True
673,25,Male,Mumbai,Bachelor,Engineering,4,Mid,59.1,6,77572.0,True,True
931,29,Female,Delhi,Master,Engineering,7,Senior,63.5,8,123475.0,True,True
878,34,Male,Chennai,High School,Sales,11,Senior,46.9,18,117463.0,False,True
304,29,Female,Chennai,Bachelor,Marketing,9,Senior,59.8,13,108258.0,True,True
421,33,Female,Mumbai,High School,Engineering,8,Senior,61.8,10,107778.0,False,True
680,45,Female,Mumbai,Bachelor,Sales,26,Manager,89.0,38,216354.0,True,True


In [2]:
# Reducing Columns for simplicity and demonstration

cols = ["Age", "Gender", "City","Education_Level", "Job_Level"]
df2 = df[cols]

In [3]:
X = df2[["Age", "Gender", "City","Education_Level"]]
y = df2[["Job_Level"]]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20)

In [5]:
# Imporing All Encoding Libraries
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

In [6]:
X_train.sample(2)

,Age,Gender,City,Education_Level
439,21,Male,Pune,Master
705,27,Female,Hyderabad,Bachelor


In [7]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(
    transformers= [
        ('tnf1', SimpleImputer(), ['Age']), # 1st column
        ('tnf2', OneHotEncoder(drop = 'first'), ['Gender', 'City']), # 2nd and 3rd column
        ('tnf3', OrdinalEncoder(categories=[["High School", "Bachelor", "Master","PhD"]]),["Education_Level"]) # 4th column
    ], remainder= 'passthrough')

In [8]:
transformer.fit_transform(X_train)
X_train = transformer.transform(X_train)
X_test = transformer.transform(X_test)
